<a href="https://colab.research.google.com/github/shashidhar078/FlipGrid/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

from sklearn.cluster import KMeans

!pip install pygeohash -q
import pygeohash as pgh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00


In [3]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

In [4]:
print(train.shape)
print(test.shape)

train.head()

(77299, 11)
(41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB


In [6]:
train.isnull().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


In [7]:
train.nunique()

,0
Index,77299
geohash,1249
day,2
timestamp,96
demand,76715
RoadType,3
NumberofLanes,5
LargeVehicles,2
Landmarks,2
Temperature,74804


**Missing Value Handling**

In [8]:
num_cols = train.select_dtypes(include=np.number).columns

In [9]:
for col in num_cols:

    train[col] = train[col].fillna(
        train[col].median()
    )

    if col in test.columns:
        test[col] = test[col].fillna(
            train[col].median()
        )

In [10]:
cat_cols = train.select_dtypes(include="object").columns

In [11]:
for col in cat_cols:

    train[col] = train[col].fillna(
        train[col].mode()[0]
    )

    if col in test.columns:
        test[col] = test[col].fillna(
            train[col].mode()[0]
        )

In [12]:
train[['hour','minute']] = (
    train['timestamp']
    .str.split(':',expand=True)
    .astype(int)
)

test[['hour','minute']] = (
    test['timestamp']
    .str.split(':',expand=True)
    .astype(int)
)

In [13]:
train['peak_hour'] = (
    ((train['hour'] >= 7) & (train['hour'] <= 10))
    |
    ((train['hour'] >= 17) & (train['hour'] <= 20))
).astype(int)

test['peak_hour'] = (
    ((test['hour'] >= 7) & (test['hour'] <= 10))
    |
    ((test['hour'] >= 17) & (test['hour'] <= 20))
).astype(int)

In [14]:
def get_time_block(hour):

    if hour < 6:
        return "night"

    elif hour < 12:
        return "morning"

    elif hour < 18:
        return "afternoon"

    else:
        return "evening"

In [15]:
train["time_block"] = train["hour"].apply(
    get_time_block
)

test["time_block"] = test["hour"].apply(
    get_time_block
)

In [16]:
train["minute_bucket"] = (
    train["minute"] // 15
)

test["minute_bucket"] = (
    test["minute"] // 15
)

In [17]:
train['hour_sin'] = np.sin(
    2*np.pi*train['hour']/24
)

train['hour_cos'] = np.cos(
    2*np.pi*train['hour']/24
)

test['hour_sin'] = np.sin(
    2*np.pi*test['hour']/24
)

test['hour_cos'] = np.cos(
    2*np.pi*test['hour']/24
)

In [18]:
def decode_geohash(x):

    lat, lon = pgh.decode(x)

    return pd.Series(
        [lat, lon]
    )

In [19]:
train[['latitude','longitude']] = (
    train['geohash']
    .apply(decode_geohash)
)

test[['latitude','longitude']] = (
    test['geohash']
    .apply(decode_geohash)
)

In [20]:
kmeans = KMeans(
    n_clusters=20,
    random_state=42
)

In [21]:
train['location_cluster'] = (
    kmeans.fit_predict(
        train[['latitude','longitude']]
    )
)

test['location_cluster'] = (
    kmeans.predict(
        test[['latitude','longitude']]
    )
)

In [22]:
center_lat = train['latitude'].mean()

center_lon = train['longitude'].mean()

In [23]:
train['distance_center'] = np.sqrt(
    (train['latitude']-center_lat)**2
    +
    (train['longitude']-center_lon)**2
)

test['distance_center'] = np.sqrt(
    (test['latitude']-center_lat)**2
    +
    (test['longitude']-center_lon)**2
)

In [24]:
train['is_rainy'] = (
    train['Weather']
    .astype(str)
    .str.lower()
    .str.contains("rain")
).astype(int)

test['is_rainy'] = (
    test['Weather']
    .astype(str)
    .str.lower()
    .str.contains("rain")
).astype(int)

In [25]:
train['temp_bin'] = pd.cut(
    train['Temperature'],
    bins=5,
    labels=False
)

test['temp_bin'] = pd.cut(
    test['Temperature'],
    bins=5,
    labels=False
)

In [26]:
train['temp_weather'] = (
    train['temp_bin'].astype(str)
    +
    "_"
    +
    train['Weather'].astype(str)
)

test['temp_weather'] = (
    test['temp_bin'].astype(str)
    +
    "_"
    +
    test['Weather'].astype(str)
)

In [27]:
train['road_lane'] = (
    train['RoadType'].astype(str)
    +
    "_"
    +
    train['NumberofLanes'].astype(str)
)

test['road_lane'] = (
    test['RoadType'].astype(str)
    +
    "_"
    +
    test['NumberofLanes'].astype(str)
)

In [28]:
vehicle_map = {
    "Yes":1,
    "No":0
}

In [29]:
train['LargeVehicles_num'] = (
    train['LargeVehicles']
    .map(vehicle_map)
)

test['LargeVehicles_num'] = (
    test['LargeVehicles']
    .map(vehicle_map)
)

In [30]:
train['lane_vehicle'] = (
    train['NumberofLanes']
    *
    train['LargeVehicles_num']
)

test['lane_vehicle'] = (
    test['NumberofLanes']
    *
    test['LargeVehicles_num']
)

In [31]:
landmark_map = {
    "Yes":1,
    "No":0
}

In [32]:
train['Landmarks_num'] = (
    train['Landmarks']
    .map(landmark_map)
)

test['Landmarks_num'] = (
    test['Landmarks']
    .map(landmark_map)
)

In [33]:
drop_cols = [
    'timestamp'
]

In [34]:
train.drop(
    columns=drop_cols,
    inplace=True
)

test.drop(
    columns=drop_cols,
    inplace=True
)

In [35]:
train.to_csv(
    "processed_train_v1.csv",
    index=False
)

test.to_csv(
    "processed_test_v1.csv",
    index=False
)

In [36]:
from google.colab import files

files.download(
    "processed_train_v1.csv"
)

files.download(
    "processed_test_v1.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>